# Experiment No. 9: Transformer-Based LLM Chatbot Using Gradio

**Name: Jamuna VP**

**Roll.NO: 24BAD038**


## Part A: Loading and Configuring a Pre-trained LLM

In this section, install the required libraries, load the model and tokenizer, tokenize a sample prompt, and generate a response using temperature and maximum output length.

In [ ]:
!pip -q install -U transformers accelerate gradio

In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

MODEL_NAME = "Qwen/Qwen2.5-0.5B-Instruct"
device = "cuda" if torch.cuda.is_available() else "cpu"

print("Device:", device)

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.float16 if device == "cuda" else torch.float32
)
model.to(device)
model.eval()

print("Model and tokenizer loaded successfully.")

Device: cpu


config.json:   0%|          | 0.00/659 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

... (2 outputs left out)

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

Model and tokenizer loaded successfully.


In [ ]:
# Prepare and tokenize a sample user prompt
sample_prompt = "Explain artificial intelligence in simple words."

sample_messages = [
    {"role": "system", "content": "You are a helpful AI assistant."},
    {"role": "user", "content": sample_prompt}
]

sample_inputs = tokenizer.apply_chat_template(
    sample_messages,
    tokenize=True,
    add_generation_prompt=True,
    return_dict=True,
    return_tensors="pt"
).to(device)

print("Sample prompt:", sample_prompt)
print("Token IDs:", sample_inputs["input_ids"])
print("Number of input tokens:", sample_inputs["input_ids"].shape[-1])

Sample prompt: Explain artificial intelligence in simple words.
Token IDs: tensor([[151644,   8948,    198,   2610,    525,    264,  10950,  15235,  17847,
             13, 151645,    198, 151644,    872,    198,    840,  20772,  20443,
          11229,    304,   4285,   4244,     13, 151645,    198, 151644,  77091,
            198]])
Number of input tokens: 28


In [ ]:
# Generate a response
with torch.no_grad():
    sample_output = model.generate(
        **sample_inputs,
        max_new_tokens=100,
        do_sample=True,
        temperature=0.7,
        top_p=0.9,
        pad_token_id=tokenizer.eos_token_id
    )

new_tokens = sample_output[0][sample_inputs["input_ids"].shape[-1]:]
sample_response = tokenizer.decode(new_tokens, skip_special_tokens=True)

print("Generated response:")
print(sample_response)

Generated response:
Artificial Intelligence (AI) is a branch of computer science that focuses on creating machines that can perform tasks that typically require human intelligence, such as learning, problem-solving, decision-making, and language understanding. AI involves the development of algorithms and models that allow computers to learn from data, understand natural language, recognize patterns, make predictions, and generate outputs similar to humans.

Key features of AI include:
1. Self-learning: The ability for an AI system to improve its performance over time without being explicitly


## Part B: Implementing Chatbot Functionality

The chatbot accepts user queries, builds a prompt using conversation history, and generates a response. Only the most recent conversation turns are included to keep the prompt within a practical length.

In [ ]:
def generate_response(user_query, conversation_history=None,
                      temperature=0.7, max_new_tokens=150):
    """Generate an answer using the loaded instruction-tuned LLM."""
    if conversation_history is None:
        conversation_history = []

    messages = [
        {"role": "system", "content": "You are a helpful, clear and concise AI assistant."}
    ]

    # Retain a limited number of previous turns for context.
    for turn in conversation_history[-4:]:
        if isinstance(turn, dict):
            role = turn.get("role")
            content = turn.get("content", "")
            if role in ("user", "assistant") and content:
                messages.append({"role": role, "content": str(content)})
        elif isinstance(turn, (list, tuple)) and len(turn) == 2:
            user_text, assistant_text = turn
            if user_text:
                messages.append({"role": "user", "content": str(user_text)})
            if assistant_text:
                messages.append({"role": "assistant", "content": str(assistant_text)})

    messages.append({"role": "user", "content": str(user_query)})

    inputs = tokenizer.apply_chat_template(
        messages,
        tokenize=True,
        add_generation_prompt=True,
        return_dict=True,
        return_tensors="pt"
    ).to(device)

    with torch.no_grad():
        output_ids = model.generate(
            **inputs,
            max_new_tokens=int(max_new_tokens),
            do_sample=True,
            temperature=max(float(temperature), 0.01),
            top_p=0.9,
            repetition_penalty=1.1,
            pad_token_id=tokenizer.eos_token_id
        )

    answer_tokens = output_ids[0][inputs["input_ids"].shape[-1]:]
    answer = tokenizer.decode(answer_tokens, skip_special_tokens=True).strip()
    return answer

# Test the chatbot function
test_query = "What is a Transformer model?"
print("User:", test_query)
print("Chatbot:", generate_response(test_query))

User: What is a Transformer model?
Chatbot: A Transformer model is a type of artificial neural network architecture designed to process and analyze sequential data, particularly sequences of words in natural language. It's widely used for tasks such as machine translation, text summarization, and question answering.

Key components of a Transformer include:

1. **Encoder Layer**: This layer processes the input sequence by taking multiple steps forward through a series of transformations. The encoder outputs a fixed-length representation that captures the context of the previous token(s).

2. **Decoder Layer**: This layer decodes the output of the encoder into a sequence of tokens. The decoder uses both past information from the encoder (via the attention mechanism) and new features learned during the training process to generate an output sequence.

3. **Self-Attention Mechan


## Part C: Developing an Interactive Interface Using Gradio

Gradio provides a chat interface that connects the user's message to the response-generation function. The interface displays messages in a chat-style layout and passes conversation history to the chatbot.

In [ ]:
import gradio as gr

def gradio_chat(message, history):
    # Gradio ChatInterface supplies the previous conversation as history.
    answer = generate_response(
        user_query=message,
        conversation_history=history,
        temperature=0.7,
        max_new_tokens=150
    )
    return answer

demo = gr.ChatInterface(
    fn=gradio_chat,
    title="Transformer-Based LLM Chatbot",
    description="Ask a question and receive a response from a pre-trained Transformer language model.",
    examples=[
        "What is artificial intelligence?",
        "Explain machine learning.",
        "What is natural language processing?"
    ]
)

demo.launch(share=True)

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://b0c5abca7f851b4bdb.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


## Part D: Application Testing and Deployment

### Test cases

Try the following prompts in the Gradio chatbot and record the actual outputs.

| Test case | Input prompt | What to observe |
|---|---|---|
| 1 | What is artificial intelligence? | Relevance and clarity |
| 2 | Explain machine learning in simple words. | Simplicity of explanation |
| 3 | What is the difference between AI and ML? | Accuracy and comparison |
| 4 | Explain a Transformer model. | Technical understanding |
| 5 | Can you summarize your previous answer? | Use of conversation history |

### Limitations to observe
- The model may occasionally produce incorrect or fabricated information (hallucination).
- Response time depends on model size, hardware and network conditions.
- The model's knowledge may not include recent events.
- Conversation context is limited by the amount of history included in the prompt.

### Create requirements.txt

In [ ]:
requirements = """torch
transformers
accelerate
gradio
"""

with open("requirements.txt", "w") as file:
    file.write(requirements)

print("requirements.txt created:")
print(requirements)

requirements.txt created:
torch
transformers
accelerate
gradio



## Result

A chatbot using a pre-trained Transformer-based LLM was implemented. The model and tokenizer were loaded using Hugging Face Transformers, responses were generated from user prompts, conversation history was incorporated, and an interactive chat interface was developed using Gradio. The application can be tested locally or in Google Colab and prepared for cloud deployment.